Data preparation 2.0

Imports:

In [ ]:
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

Set filepaths:

In [ ]:
train_file_path = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_pre files\X_train.h5"

test_file_path = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_pre files\X_test.h5"

train_labels_path = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_pre files\y_train_tX9Br0C.csv"

CSV_META = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_pre files\X_train_h7ipJUo.csv"


Set variables:

In [ ]:
FS = 100
WIN_SEC = 90
SAMPLES_PER_WIN = FS * WIN_SEC   # 9000
N_CH = 8
N_WINS_EXPECTED = 200


N_WINS = 200
T_1HZ_PER_NIGHT = N_WINS * WIN_SEC  # 18000

Load files and split by subject ID to get whole night data

In [ ]:
def load_and_split_by_subject(h5_path, n_wins_expected=200, sort_by_id=True):
    with h5py.File(h5_path, "r") as f:
        raw = f["data"][:]   # (N, 72002)

    ids      = raw[:, 0].astype(int)
    subjects = raw[:, 1].astype(int)

    X_flat = raw[:, 2:].astype("float32")     # (N, 72000)
    X = X_flat.reshape(len(X_flat), SAMPLES_PER_WIN, N_CH)  # (N, 9000, 8)

    # agrupa por sujeito
    subject_to_indices = {}
    for i, s in enumerate(subjects):
        subject_to_indices.setdefault(int(s), []).append(i)

    # monta dict final
    nights = {}
    for subj, idxs in subject_to_indices.items():
        idxs = np.array(idxs)

        if sort_by_id:
            order = np.argsort(ids[idxs])
            idxs = idxs[order]

        # sanity checks
        if len(idxs) != n_wins_expected:
            print(f"[WARN] subject {subj}: {len(idxs)} janelas (esperado {n_wins_expected})")

        X_subj = X[idxs]  # (n_win, 9000, 8)

        nights[subj] = {
            "ids": ids[idxs],                 # window ids em ordem
            "X_win": X_subj,                  # (n_win, 9000, 8)
            "X_flat": X_subj.reshape(-1, N_CH)  # (n_win*9000, 8)
        }

    return nights


def load_y_by_id(y_csv_path):
    y_df = pd.read_csv(y_csv_path)
    mask_cols = [c for c in y_df.columns if c.startswith("y_")]
    y_df = y_df.set_index("ID")
    return y_df[mask_cols].astype("int8")

def attach_y_to_nights(nights, y_by_id_df):
    for subj in nights:
        ids_subj = nights[subj]["ids"]
        y_win = y_by_id_df.loc[ids_subj].to_numpy()        # (n_win, 90)
        nights[subj]["y_win"]  = y_win
        nights[subj]["y_flat"] = y_win.reshape(-1)         # (n_win*90,)
    return nights

In [ ]:
train_nights = load_and_split_by_subject(train_file_path)
test_nights  = load_and_split_by_subject(test_file_path)

print("# subjects train:", len(train_nights))
print("# subjects test:", len(test_nights))
print("Example shapes subject 0:",
      train_nights[list(train_nights.keys())[0]]["X_win"].shape,
      train_nights[list(train_nights.keys())[0]]["X_flat"].shape)

y_by_id = load_y_by_id(train_labels_path)
train_nights = attach_y_to_nights(train_nights, y_by_id)

# now a subject has:
# X_win (200,9000,8), X_flat (1_800_000,8)
# y_win (200,90),     y_flat (18_000,)


In [ ]:
def build_night_tensor(h5_path):
    with h5py.File(h5_path, "r") as f:
        raw = f["data"][:]

    ids      = raw[:, 0].astype(int)
    subjects = raw[:, 1].astype(int)

    X_flat = raw[:, 2:].astype("float32")
    X = X_flat.reshape(len(X_flat), SAMPLES_PER_WIN, N_CH)

    nights = []
    subj_list = []

    for subj in np.unique(subjects):
        idxs = np.where(subjects == subj)[0]
        idxs = idxs[np.argsort(ids[idxs])]     # assures temporal order

        X_subj = X[idxs]                       # (200, 9000, 8)
        X_night = X_subj.reshape(-1, N_CH)     # (1_800_000, 8)

        nights.append(X_night)
        subj_list.append(int(subj))

    return np.stack(nights), np.array(subj_list)

# --- usage ---
X_train_nights, subj_train_ids = build_night_tensor(train_file_path)
X_test_nights,  subj_test_ids  = build_night_tensor(test_file_path)

print("X_train_nights:", X_train_nights.shape)  # (22, 1800000, 8)
print("X_test_nights: ", X_test_nights.shape)   # (22, 1800000, 8)

Xn_train_nights, mean_train_n, std_train_n = normalize_per_night(X_train_nights)
Xn_test_nights,  mean_test_n,  std_test_n  = normalize_per_night(X_test_nights)

print("Xn_train_nights:", Xn_train_nights.shape)
print("Xn_test_nights: ", Xn_test_nights.shape)

normalize per night and per channel (mean/std along time T)

In [ ]:
def normalize_per_night(X_nights, eps=1e-8):
    """
    X_nights: (n_nights, T, C) ex: (22, 1_800_000, 8)
    normalizes per night and per channel (mean/std along time T)
    """
    mean = X_nights.mean(axis=1, keepdims=True)          # (n_nights, 1, C)
    std  = X_nights.std(axis=1, keepdims=True) + eps     # (n_nights, 1, C)
    Xn = (X_nights - mean) / std
    return Xn, mean.squeeze(1), std.squeeze(1)           # mean/std: (n_nights, C)


In [ ]:
# mean ~0 and std ~1 per night/channel
mean_train_check = Xn_train_nights.mean(axis=1)   # (22, 8)
std_train_check = Xn_train_nights.std(axis=1)    # (22, 8)

print("mean train abs max:", np.max(np.abs(mean_train_check)))
print("std train min/max:", std_train_check.min(), std_train_check.max())

mean_test_check = Xn_test_nights.mean(axis=1)   # (22, 8)
std_test_check = Xn_test_nights.std(axis=1)    # (22, 8)

print("mean test abs max:", np.max(np.abs(mean_test_check)))
print("std test min/max:", std_test_check.min(), std_test_check.max())


set y train to night shape

In [ ]:
# 1) IDs per subject (from meta CSV)
meta = pd.read_csv(CSV_META)
meta = meta.sort_values(["Subject_ID", "ID"])

ids_by_subj = {
    int(s): g["ID"].to_numpy().astype(int)
    for s, g in meta.groupby("Subject_ID")
}

print("n_subjects:", len(ids_by_subj))
print("example subj:", list(ids_by_subj.keys())[:3], "len:", [len(ids_by_subj[k]) for k in list(ids_by_subj.keys())[:3]])

# 2) y per ID (from labels CSV)
y_df = pd.read_csv(train_labels_path)
mask_cols = [c for c in y_df.columns if c.startswith("y_")]
y_by_id = y_df.set_index("ID")[mask_cols].astype("int8")

# 3) build y_nights
y_nights = []
subj_order = sorted(ids_by_subj.keys())

for s in subj_order:
    ids_s = ids_by_subj[s]
    y_win = y_by_id.loc[ids_s].to_numpy()   # (200, 90)
    y_nights.append(y_win.reshape(-1))      # (18000,)

y_nights = np.stack(y_nights)              # (22, 18000)

print("y_nights:", y_nights.shape)
print("unique vals:", np.unique(y_nights))
print("% ones:", y_nights.mean() * 100)



split train and val sets before creating chunks

In [ ]:
SEED = 42
rng = np.random.default_rng(SEED)
idx = np.arange(22)
rng.shuffle(idx)

n_train = int(0.7 * len(idx))  # 15
train_idx = idx[:n_train]
val_idx   = idx[n_train:]

X_trainN = Xn_train_nights[train_idx]
y_trainN = y_nights[train_idx]
X_valN   = Xn_train_nights[val_idx]
y_valN   = y_nights[val_idx]

train_ds = make_tf_dataset(X_trainN, y_trainN, chunk_sec=300, stride_sec=60, batch_size=4)
val_ds   = make_tf_dataset(X_valN,   y_valN,   chunk_sec=300, stride_sec=60, batch_size=4, shuffle_buffer=256)


creates chunked TF datasets for training/validation

In [ ]:
def chunk_generator(X_nights, y_nights, chunk_sec=300, stride_sec=60, fs=100):
    chunk_len = chunk_sec * fs
    stride_len = stride_sec * fs

    n_nights, T, C = X_nights.shape
    assert y_nights.shape[0] == n_nights
    assert y_nights.shape[1] == T // fs, f"y needs to have {T//fs} steps (1Hz) per night"

    for n in range(n_nights):
        X = X_nights[n]   # (T, 8) at 100Hz
        y = y_nights[n]   # (T//100,) at 1Hz
        for start in range(0, T - chunk_len + 1, stride_len):
            end = start + chunk_len

            X_chunk = X[start:end]
            y_chunk = y[start//fs : end//fs]

            # sanity: garante alinhamento perfeito
            if y_chunk.shape[0] != chunk_sec:
                continue

            yield X_chunk.astype("float32"), y_chunk.astype("float32")


In [ ]:
#overlap = chunk − stride = 300 − 60 = 240 seconds 80% overlap
# stride = 120s → overlap = 180s (60%)
# stride = 150s → overlap = 150s (50%)
# stride = 300s → overlap = 0 (no overlap, but worse coverage)
CHUNK_SEC = 300  #5-minute (300s) windows 
STRIDE_SEC = 60
FS = 100

gen = chunk_generator(Xn_train_nights, y_nights, chunk_sec=CHUNK_SEC, stride_sec=STRIDE_SEC, fs=FS)
Xc, yc = next(gen)
print(Xc.shape, yc.shape)          # (30000, 8) (300,)
print("chunk % ones:", yc.mean()*100)


In [ ]:


def make_tf_dataset(X_nights, y_nights, chunk_sec=300, stride_sec=60, fs=100,
                    batch_size=4, shuffle_buffer=1024):
    chunk_len = chunk_sec * fs

    output_signature = (
        tf.TensorSpec(shape=(chunk_len, 8), dtype=tf.float32),
        tf.TensorSpec(shape=(chunk_sec,), dtype=tf.float32),
    )

    ds = tf.data.Dataset.from_generator(
        lambda: chunk_generator(X_nights, y_nights, chunk_sec, stride_sec, fs),
        output_signature=output_signature
    )

    ds = ds.shuffle(shuffle_buffer, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


In [ ]:
CHUNK_SEC = 300
STRIDE_SEC = 60
BATCH_SIZE = 4

train_ds = make_tf_dataset(Xn_train_nights, y_nights, chunk_sec=CHUNK_SEC, stride_sec=STRIDE_SEC, batch_size=BATCH_SIZE)

for xb, yb in train_ds.take(1):
    print("batch X:", xb.shape, "batch y:", yb.shape)  # (4, 30000, 8) e (4, 300)
